In [2]:
import os
import subprocess
import time
import json
import requests
from pydantic import BaseModel, Field

# =====================================================================
# PHASE 1: SYSTEM DEPENDENCY & OLLAMA DAEMON INITIALIZATION
# =====================================================================
print("[SYSTEM] Updating package lists...")
subprocess.run("apt-get update -y > /dev/null 2>&1", shell=True)

print("[SYSTEM] Installing zstd extraction dependency...")
subprocess.run("apt-get install zstd -y > /dev/null 2>&1", shell=True)

print("[SYSTEM] Fetching and installing Ollama binary...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)

print("[SYSTEM] Purging legacy zombie processes...")
subprocess.run("pkill ollama", shell=True)
time.sleep(2)

print("[SYSTEM] Booting Ollama server as a detached daemon...")
# os.setsid detaches the process group so Kaggle cannot reap it mid-cell
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("ollama_stdout.log", "w"),
    stderr=open("ollama_stderr.log", "w"),
    preexec_fn=os.setsid
)

print("[SYSTEM] Awaiting port 11434 binding verification...")
server_online = False
for _ in range(30):
    try:
        r = requests.get("http://localhost:11434", timeout=1)
        if r.status_code == 200:
            server_online = True
            print("[SYSTEM] Ollama daemon successfully bound to port 11434.")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

if not server_online:
    print("[FATAL] Ollama failed to initialize. Dumping stderr logs:")
    if os.path.exists("ollama_stderr.log"):
        with open("ollama_stderr.log", "r") as f:
            print(f.read())
    raise RuntimeError("Daemon initialization aborted.")

print("[SYSTEM] Pulling Llama-3 8B model weights into VRAM...")
subprocess.run(["ollama", "pull", "llama3"], check=True)
print("[SYSTEM] Hardware and weights validated. Starting tagging loop immediately.\n" + "="*70)
import os
import time
import json
import requests
import re
from pydantic import BaseModel, Field

# =====================================================================
# PHASE 2: YOUR EXACT TAGGING LOGIC (OPTIMIZED FOR FAILOVER)
# =====================================================================
ALLOWED_MOODS = [
    "aggressive", "confident", "melancholic", "introspective", "motivational", 
    "hype", "dark", "romantic", "vulnerable", "rebellious", "playful", 
    "emotional", "storytelling"
]

ALLOWED_GENRES = [
    "trap", "boom_bap", "drill", "grime", "conscious_rap", "melodic_rap", 
    "experimental", "underground", "mainstream", "alternative_hiphop"
]

ALLOWED_RHYME_SCHEMES = [
    "monorhyme", "couplet", "alternate_rhyme", "internal_rhyme", 
    "multisyllabic", "dense_internal", "freeform"
]

ALLOWED_CADENCES = [
    "slow", "midtempo", "fast", "switching", "staccato", "rolling", "bouncy"
]

class Annotations(BaseModel):
    mood: str = Field(description=f"A single mood strictly from the allowed list: {ALLOWED_MOODS}")
    genre: str = Field(description=f"A single genre strictly from the allowed list: {ALLOWED_GENRES}")
    rhyme_scheme: str = Field(description=f"A single rhyme scheme strictly from the allowed list: {ALLOWED_RHYME_SCHEMES}")
    cadence: str = Field(description=f"A single cadence strictly from the allowed list: {ALLOWED_CADENCES}")
    year: int = Field(description="The accurate release year of the song, determined based on the name of the song and artist.")

class TaggingResult(BaseModel):
    annotations: Annotations

def tag_song_with_gemma4(title: str, artist: str, lyrics: str, max_retries=5):
    truncated_lyrics = lyrics[:2500]
    
    prompt = (
        f"Analyze this rap song: '{title}' by {artist}. "
        f"You MUST select exactly 1 primary mood from: {ALLOWED_MOODS}. "
        f"You MUST select exactly 1 genre from: {ALLOWED_GENRES}. "
        f"You MUST select exactly 1 rhyme scheme from: {ALLOWED_RHYME_SCHEMES}. "
        f"You MUST select exactly 1 cadence from: {ALLOWED_CADENCES}. "
        f"You MUST provide the accurate release year of the song based on the name of the song and artist. "
        "Do not invent any new tags. Your output must strictly conform to the provided schema:\n"
        '{"annotations": {"mood": "...", "genre": "...", "rhyme_scheme": "...", "cadence": "...", "year": 0}}'
    )
    
    system_instruction = (
        "You are an expert musicologist with 30 years of experience in rap songs and a meticulous tagger. "
        "You must evaluate the song and output the annotations strictly from the allowed lists. "
        "Your output must strictly conform to the provided JSON schema."
    )
    
    payload = {
        "model": "llama3",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": f"{prompt}\n\nLyrics:\n{truncated_lyrics}"}
        ],
        "format": "json",
        "stream": False
    }
    
    for attempt in range(max_retries):
        try:
            # Extended timeout to prevent VRAM allocation drops
            response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=120)
            response.raise_for_status()
            
            cleaned_text = response.json().get("message", {}).get("content", "").strip()
            
            # Isolate JSON dictionary explicitly using regex boundaries
            match = re.search(r'\{.*\}', cleaned_text, re.DOTALL)
            if match:
                cleaned_text = match.group(0)
            
            result_data = json.loads(cleaned_text)
            return result_data
            
        except Exception as e:
            print(f"Error calling local Ollama endpoint (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                sleep_time = (2 ** attempt) * 5
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                print("Max retries reached. Returning None.")
                return None

# --- Main Processing Loop ---
input_filename = '/kaggle/input/datasets/namanatgoel/diverse-rap-corpus-500/diverse_rap_corpus_500.json' 
output_filename = '/kaggle/working/tagged_rap_dataset_gemma4.json'

with open(input_filename, 'r', encoding='utf-8') as f:
    songs = json.load(f)

print(f"Loaded {len(songs)} songs. Batch tagging via local Ollama...")

final_corpus = []
processed_titles = set()

if os.path.exists(output_filename):
    with open(output_filename, 'r', encoding='utf-8') as f:
        final_corpus = json.load(f)
        for song in final_corpus:
            processed_titles.add((song['song'].lower().strip(), song['artist'].lower().strip()))
    print(f"Resuming from {len(final_corpus)} previously tagged songs.")

for i, song in enumerate(songs):
    title_clean = song['title'].lower().strip()
    artist_clean = song['artist'].lower().strip()
    if (title_clean, artist_clean) in processed_titles:
        print(f"[{i+1}/{len(songs)}] Skipping already tagged: {song['title']} by {song['artist']}")
        continue

    print(f"[{i+1}/{len(songs)}] Tagging: {song['title']} by {song['artist']}...")
    
    result = tag_song_with_gemma4(song['title'], song['artist'], song['lyrics'])
    
    if result:
        annotations = result.get("annotations", {})
        fetched_year = annotations.pop("year", None)

        final_song = {
            "artist": song['artist'],
            "song": song['title'],
            "year": fetched_year,
            "lyrics": song['lyrics'],
            "annotations": annotations
        }
        final_corpus.append(final_song)
        processed_titles.add((title_clean, artist_clean))
        
        # State Checkpoint written safely to disk
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(final_corpus, f, indent=4)
    else:
        print(f"Skipping {song['title']} due to error.")
    
    time.sleep(1) 

print(f"\nSuccessfully finished! Total {len(final_corpus)} songs successfully tagged.")
print(f"FILE SAVED AT: {output_filename}")
'''
# =====================================================================
# PHASE 2: YOUR EXACT TAGGING LOGIC (UNALTERED)
# =====================================================================
ALLOWED_MOODS = [
    "aggressive", "confident", "melancholic", "introspective", "motivational", 
    "hype", "dark", "romantic", "vulnerable", "rebellious", "playful", 
    "emotional", "storytelling"
]

ALLOWED_GENRES = [
    "trap", "boom_bap", "drill", "grime", "conscious_rap", "melodic_rap", 
    "experimental", "underground", "mainstream", "alternative_hiphop"
]

ALLOWED_RHYME_SCHEMES = [
    "monorhyme", "couplet", "alternate_rhyme", "internal_rhyme", 
    "multisyllabic", "dense_internal", "freeform"
]

ALLOWED_CADENCES = [
    "slow", "midtempo", "fast", "switching", "staccato", "rolling", "bouncy"
]

class Annotations(BaseModel):
    mood: str = Field(description=f"A single mood strictly from the allowed list: {ALLOWED_MOODS}")
    genre: str = Field(description=f"A single genre strictly from the allowed list: {ALLOWED_GENRES}")
    rhyme_scheme: str = Field(description=f"A single rhyme scheme strictly from the allowed list: {ALLOWED_RHYME_SCHEMES}")
    cadence: str = Field(description=f"A single cadence strictly from the allowed list: {ALLOWED_CADENCES}")
    year: int = Field(description="The accurate release year of the song, determined based on the name of the song and artist.")

class TaggingResult(BaseModel):
    annotations: Annotations

def tag_song_with_gemma4(title: str, artist: str, lyrics: str, max_retries=5):
    truncated_lyrics = lyrics[:2500]
    
    prompt = (
        f"Analyze this rap song: '{title}' by {artist}. "
        f"You MUST select exactly 1 primary mood from: {ALLOWED_MOODS}. "
        f"You MUST select exactly 1 genre from: {ALLOWED_GENRES}. "
        f"You MUST select exactly 1 rhyme scheme from: {ALLOWED_RHYME_SCHEMES}. "
        f"You MUST select exactly 1 cadence from: {ALLOWED_CADENCES}. "
        f"You MUST provide the accurate release year of the song based on the name of the song and artist. "
        "Do not invent any new tags. Your output must strictly conform to the provided schema:\n"
        '{"annotations": {"mood": "...", "genre": "...", "rhyme_scheme": "...", "cadence": "...", "year": 0}}'
    )
    
    system_instruction = (
        "You are an expert musicologist with 30 years of experience in rap songs and a meticulous tagger. "
        "You must evaluate the song and output the annotations strictly from the allowed lists. "
        "Your output must strictly conform to the provided JSON schema."
    )
    
    payload = {
        "model": "llama3",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": f"{prompt}\n\nLyrics:\n{truncated_lyrics}"}
        ],
        "format": "json",
        "stream": False
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=60)
            response.raise_for_status()
            
            cleaned_text = response.json().get("message", {}).get("content", "").strip()
            
            if cleaned_text.startswith("```json"):
                cleaned_text = cleaned_text[7:]
            elif cleaned_text.startswith("```"):
                cleaned_text = cleaned_text[3:]
            if cleaned_text.endswith("```"):
                cleaned_text = cleaned_text[:-3]
            cleaned_text = cleaned_text.strip()

            result_data = json.loads(cleaned_text)
            return result_data
            
        except Exception as e:
            print(f"Error calling local Ollama endpoint (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                sleep_time = (2 ** attempt) * 5
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                print("Max retries reached. Returning None.")
                return None

# --- Main Processing Loop ---
input_filename = '/kaggle/input/datasets/namanatgoel/diverse-rap-corpus-500/diverse_rap_corpus_500.json' 
output_filename = '/kaggle/working/tagged_rap_dataset_gemma4.json'

with open(input_filename, 'r', encoding='utf-8') as f:
    songs = json.load(f)

print(f"Loaded {len(songs)} songs. Batch tagging via local Ollama...")

final_corpus = []
processed_titles = set()

if os.path.exists(output_filename):
    with open(output_filename, 'r', encoding='utf-8') as f:
        final_corpus = json.load(f)
        for song in final_corpus:
            processed_titles.add((song['song'].lower().strip(), song['artist'].lower().strip()))
    print(f"Resuming from {len(final_corpus)} previously tagged songs.")

for i, song in enumerate(songs):
    title_clean = song['title'].lower().strip()
    artist_clean = song['artist'].lower().strip()
    if (title_clean, artist_clean) in processed_titles:
        print(f"[{i+1}/{len(songs)}] Skipping already tagged: {song['title']} by {song['artist']}")
        continue

    print(f"[{i+1}/{len(songs)}] Tagging: {song['title']} by {song['artist']}...")
    
    result = tag_song_with_gemma4(song['title'], song['artist'], song['lyrics'])
    
    if result:
        annotations = result.get("annotations", {})
        fetched_year = annotations.pop("year", None)

        final_song = {
            "artist": song['artist'],
            "song": song['title'],
            "year": fetched_year,
            "lyrics": song['lyrics'],
            "annotations": annotations
        }
        final_corpus.append(final_song)
        processed_titles.add((title_clean, artist_clean))
        
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(final_corpus, f, indent=4)
    else:
        print(f"Skipping {song['title']} due to error.")
    
    time.sleep(1) # Base throttling reduced since latency is internal

print(f"\nSuccessfully finished! Total {len(final_corpus)} songs successfully tagged.")'''

[SYSTEM] Updating package lists...
[SYSTEM] Installing zstd extraction dependency...
[SYSTEM] Fetching and installing Ollama binary...
[SYSTEM] Purging legacy zombie processes...
[SYSTEM] Booting Ollama server as a detached daemon...
[SYSTEM] Awaiting port 11434 binding verification...
[SYSTEM] Ollama daemon successfully bound to port 11434.
[SYSTEM] Pulling Llama-3 8B model weights into VRAM...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏ 649 KB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   2% ▕                  ▏  70 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 125 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   5% ▕                  ▏ 226 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   7% ▕█                 ▏ 325 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   9% ▕█                 ▏ 427 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:  10% ▕█                 ▏ 475 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:  12% ▕██                ▏ 560 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:  14% ▕██                ▏ 648 MB/4.7 GB            

[SYSTEM] Hardware and weights validated. Starting tagging loop immediately.
Loaded 500 songs. Batch tagging via local Ollama...
[1/500] Tagging: Doomsday by MF DOOM...
[2/500] Tagging: Rapp Snitch Knishes by MF DOOM...
[3/500] Tagging: Beef Rapp by MF DOOM...
[4/500] Tagging: One Beer by MF DOOM...
[5/500] Tagging: Hoe Cakes by MF DOOM...
[6/500] Tagging: Rhymes Like Dimes by MF DOOM...
[7/500] Tagging: That’s That by MF DOOM...
[8/500] Tagging: Deep Fried Frenz by MF DOOM...
[9/500] Tagging: All Outta Ale by MF DOOM...
[10/500] Tagging: Potholderz by MF DOOM...
[11/500] Tagging: Gazzillion Ear by MF DOOM...
[12/500] Tagging: Vomitspit by MF DOOM...
[13/500] Tagging: ? (Question Mark) by MF DOOM...
[14/500] Tagging: Kon Karne by MF DOOM...
[15/500] Tagging: Kookies (2007) by MF DOOM...
[16/500] Tagging: Kon Queso by MF DOOM...
[17/500] Tagging: Gas Drawls by MF DOOM...
[18/500] Tagging: Batty Boyz by MF DOOM...
[19/500] Tagging: Guinnessez by MF DOOM...
[20/500] Tagging: Red and Gold b

'\n# =====================================================================\n# PHASE 2: YOUR EXACT TAGGING LOGIC (UNALTERED)\n# =====================================================================\nALLOWED_MOODS = [\n    "aggressive", "confident", "melancholic", "introspective", "motivational", \n    "hype", "dark", "romantic", "vulnerable", "rebellious", "playful", \n    "emotional", "storytelling"\n]\n\nALLOWED_GENRES = [\n    "trap", "boom_bap", "drill", "grime", "conscious_rap", "melodic_rap", \n    "experimental", "underground", "mainstream", "alternative_hiphop"\n]\n\nALLOWED_RHYME_SCHEMES = [\n    "monorhyme", "couplet", "alternate_rhyme", "internal_rhyme", \n    "multisyllabic", "dense_internal", "freeform"\n]\n\nALLOWED_CADENCES = [\n    "slow", "midtempo", "fast", "switching", "staccato", "rolling", "bouncy"\n]\n\nclass Annotations(BaseModel):\n    mood: str = Field(description=f"A single mood strictly from the allowed list: {ALLOWED_MOODS}")\n    genre: str = Field(descripti